## 쿠텐 top10 상품 추출 및 모든 리뷰데이터 추출 코드

In [2]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import time
import random
import json
import re
from datetime import datetime

# ── 설정 ──────────────────────────────────────────────────────────
RANK_SAVE_FILE   = "qoo10_rankings_current.jsonl"  
REVIEW_SAVE_FILE = "qoo10_reviews_master.jsonl"    

# ── 함수: 단일 상품 리뷰 전체 수집 (사용자 제공 로직 유지) ──────────────
def get_qoo10_reviews_all(gd_no, product_name):
    all_reviews = []
    page     = 1
    base_url = "https://www.qoo10.jp/gmkt.inc/Goods/GoodsReviewAjaxAppend.aspx"

    headers = {
        "User-Agent"     : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
        "Accept"         : "text/html, */*",
        "Accept-Language": "ja,en-US;q=0.9,en;q=0.8,ko;q=0.7",
        "Referer"        : f"https://www.qoo10.jp/item/x/{gd_no}",
        "X-Requested-With": "XMLHttpRequest",
    }

    print(f"\n   🚀 [{product_name[:30]}...] 리뷰 수집 시작...")

    while True:
        params = {
            "gd_no"             : gd_no,
            "group_code"        : "2",
            "page_no"           : page,
            "page_size"         : "100",
            "sort_type"         : "P",
            "contents_cnt"      : "0",
            "___cache_expire___": int(time.time() * 1000),
        }
        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=15)
            if response.status_code != 200: break

            html_content = response.text.strip()
            if not html_content: break

            soup = BeautifulSoup(html_content, "html.parser")
            review_items = soup.find_all("li", recursive=False) or soup.select("li")
            if not review_items: break

            for item in review_items:
                txt_tag = item.select_one(".review_txt")
                if not txt_tag: continue

                content   = txt_tag.get_text(strip=True)
                score_tag = item.select_one(".score")
                rating    = score_tag.get_text(strip=True) if score_tag else "5"
                user_info = item.select_one(".review_user_info")
                user_meta = user_info.get_text(" | ", strip=True) if user_info else ""
                type_tag  = item.select_one(".review_user_type")
                skin_info = type_tag.get_text(strip=True) if type_tag else ""

                all_reviews.append({
                    "gd_no"   : gd_no,
                    "Page"    : page,
                    "Rating"  : rating,
                    "Review"  : content,
                    "UserInfo": user_meta,
                    "SkinType": skin_info,
                })

            print(f"   🔄 {page}페이지 수집 중... (누적: {len(all_reviews)}개)", end="\r")
            page += 1
            time.sleep(0.5)

        except Exception as e:
            print(f"\n   ❌ 리뷰 에러: {e}")
            break

    print(f"\n   ✨ 리뷰 완료: {len(all_reviews)}개")
    return all_reviews


# ── 메인 실행부 ──────────────────────────────────────────────────
options = uc.ChromeOptions()
options.add_argument('--headless') 
options.add_argument('--window-size=1920,1080')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--incognito')
options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')

driver = None
try:
    driver = uc.Chrome(options=options)
    # 카테고리 베이스 링크
    target_url = "https://www.qoo10.jp/cat/120000012"

    print(f"📡 큐텐 카테고리 접속 중: {target_url}")
    driver.get(target_url)
    time.sleep(random.uniform(6, 9))

    # 리스트 로딩을 위해 스크롤 내림
    driver.execute_script("window.scrollTo(0, 2000);")
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    # li 구조의 상품 리스트 선택
    items = soup.select('ul#search_result_item_list > li')

    rank_data_list = []
    all_review_master = []
    rank_count = 1

    for item in items:
        if rank_count > 10: break

        try:
            # 1. PR 상품 제외 (ad_cps 클래스 또는 PR 배지 확인)
            is_pr = item.select_one(".ad_cps, .icon_pr, .item_pr") or "PR" in item.get_text()
            if is_pr: continue

            # 2. 리뷰 개수 확인 (리뷰 없는 상품 제외)
            review_tag = item.select_one(".review_total_count")
            reviews_raw = review_tag.get_text(strip=True) if review_tag else "0"
            reviews_num = int(re.sub(r'[^0-9]', '', reviews_raw)) if reviews_raw != "0" else 0
            
            if reviews_num == 0: continue # 리뷰 없으면 패스

            # 3. 상품 기본 정보 추출
            goods_code = item.get("goodscode")
            brand_tag  = item.select_one(".txt_brand")
            brand      = brand_tag.get_text(strip=True).replace("公式", "").strip() if brand_tag else "N/A"
            
            title_tag  = item.select_one("a.tt")
            title      = title_tag.get_text(strip=True) if title_tag else "N/A"
            
            price_tag  = item.select_one(".prc strong")
            price_str  = price_tag.get_text(strip=True) if price_tag else "0"

            # 평점 (별점 %를 5점 만점으로 변환)
            rating_star = item.select_one(".review_rating_star")
            rating = 0.0
            if rating_star and "style" in rating_star.attrs:
                width_match = re.search(r'width:\s*(\d+)%', rating_star["style"])
                if width_match:
                    rating = round(float(width_match.group(1)) / 20, 1)

            # 4. 요구하신 Rakuten 스타일 포맷팅
            product_info = {
                "rank": rank_count,
                "title": title,
                "shop_name": brand,
                "rating": rating,
                "reviews": reviews_num,
                "price": price_str,
                "trend": "Stay",
                "url": f"https://www.qoo10.jp/item/x/{goods_code}",
                "shop_id": "", # 큐텐 구조상 빈값 또는 브랜드번호 파싱 필요
                "item_id": goods_code,
                "platform": "Qoo10",
                "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }

            print(f"\n📍 {rank_count}위 확정: [{brand}] {goods_code}")
            rank_data_list.append(product_info)

            # 5. 해당 상품의 모든 리뷰 수집 (AJAX)
            if goods_code:
                reviews_data = get_qoo10_reviews_all(goods_code, title)
                all_review_master.extend(reviews_data)
            
            rank_count += 1

        except Exception as e:
            continue

    # ── 저장 (JSONL 형식) ──
    with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
        for entry in rank_data_list:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
        for review in all_review_master:
            f.write(json.dumps(review, ensure_ascii=False) + "\n")

    print(f"\n✨ 수집 완료: 상품 {len(rank_data_list)}개 / 총 리뷰 {len(all_review_master)}개")

except Exception as e:
    print(f"❌ 치명적 오류: {e}")
finally:
    if driver:
        driver.quit()

📡 큐텐 카테고리 접속 중: https://www.qoo10.jp/cat/120000012

📍 1위 확정: [DOPAMY] 1184035014

   🚀 [【公式】【1+1】日本初上陸！ Qoo10限定 ２つ自分で選...] 리뷰 수집 시작...
   🔄 4페이지 수집 중... (누적: 307개)
   ✨ 리뷰 완료: 307개

📍 2위 확정: [DOPAMY] 1172499524

   🚀 [【公式】日本初上陸！ ニューロペップ８R アンプルセラム 3...] 리뷰 수집 시작...
   🔄 8페이지 수집 중... (누적: 709개)
   ✨ 리뷰 완료: 709개

📍 3위 확정: [be bare] 1180516822

   🚀 [くるくる クレンジングバーム50ml...] 리뷰 수집 시작...
   🔄 3페이지 수집 중... (누적: 259개)
   ✨ 리뷰 완료: 259개

📍 4위 확정: [be bare] 1195669744

   🚀 [くるくるクレンジングバーム 50ml ＋ レフィル 50ml...] 리뷰 수집 시작...
   🔄 1페이지 수집 중... (누적: 5개)
   ✨ 리뷰 완료: 5개

📍 5위 확정: [ザツールラボ] 855174218

   🚀 [701 洗顔ブラシクレンジング ブラシ S...] 리뷰 수집 시작...
   🔄 1페이지 수집 중... (누적: 12개)
   ✨ 리뷰 완료: 12개

📍 6위 확정: [ザツールラボ] 855171577

   🚀 [1001 洗顔フォーム スウィッピング フェイス クレンザー...] 리뷰 수집 시작...
   🔄 1페이지 수집 중... (누적: 2개)
   ✨ 리뷰 완료: 2개

📍 7위 확정: [DOPAMY] 1175080093

   🚀 [【公式】日本初上陸！ 5点セット ニューロペップ８R　 クレ...] 리뷰 수집 시작...
   🔄 2페이지 수집 중... (누적: 157개)
   ✨ 리뷰 완료: 157개

📍 8위 확정: [DOPAMY] 1175081244

   🚀 [【公式】日本初上陸！ コンプリート6点セット ニューロペップ...

# 쿠텐 번역 코드

In [2]:
import json
import time
import re
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from deep_translator import GoogleTranslator

# ── 설정 ──────────────────────────────────────────────────────────
INPUT_FILE  = './qoo10_reviews_master.jsonl'
OUTPUT_FILE = 'qoo10_master_translated_jp_ko.jsonl'
BODY_COL    = 'Review'    # 번역할 리뷰 본문 키

# [병렬 처리 설정]
CHUNK_SIZE  = 5      # 묶음 번역 단위
MAX_WORKERS = 4     # 스레드 수 (안정성을 위해 2 권장)
MAX_RETRIES = 3      # 실패 시 재시도
RETRY_SLEEP = 3.0    # 재시도 간격
CHUNK_DELAY = 1.0    # 번역 단계별 대기
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_numbered(texts: list) -> str:
    return "\n".join(f"[{i+1}] {str(t).strip()}" for i, t in enumerate(texts))

def parse_numbered(text: str, expected_n: int) -> list:
    pattern = re.compile(r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|$)', re.DOTALL)
    found = pattern.findall(text)
    result = {int(idx): body.strip() for idx, body in found}
    return [result.get(i + 1, "") for i in range(expected_n)]

def translate_single(text: str, src: str, tgt: str) -> str:
    if not text or not str(text).strip(): return ""
    for attempt in range(MAX_RETRIES):
        try:
            res = GoogleTranslator(source=src, target=tgt).translate(text)
            if res: return res.strip()
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return "번역실패"

def process_chunk(texts: list, src: str, tgt: str) -> list:
    if not texts: return []
    joined = build_numbered(texts)
    for attempt in range(MAX_RETRIES):
        try:
            translated = GoogleTranslator(source=src, target=tgt).translate(joined)
            if not translated: raise ValueError("응답 없음")
            parts = parse_numbered(translated, len(texts))
            if all(p.strip() for p in parts): return parts
            for i, p in enumerate(parts):
                if not p: parts[i] = translate_single(texts[i], src, tgt)
            return parts
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return [translate_single(t, src, tgt) for t in texts]

def translate_workflow(texts: list):
    """ja -> en -> ko 2단계 번역"""
    # 1단계: 일 -> 영
    en_texts = process_chunk(texts, 'ja', 'en')
    time.sleep(CHUNK_DELAY)
    # 2단계: 영 -> 한
    ko_texts = process_chunk(en_texts, 'en', 'ko')
    return ko_texts

def main():
    print(f"📥 Qoo10 데이터 로딩 중: {INPUT_FILE}")
    records = []
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip(): records.append(json.loads(line))
    except FileNotFoundError:
        print("❌ 입력 파일을 찾을 수 없습니다.")
        return

    df = pd.DataFrame(records)
    total_len = len(df)
    print(f"✅ 총 {total_len:,}건 확인 (본문 번역만 진행)")

    # 리뷰 본문(Review) 번역
    bodies = df[BODY_COL].fillna("").tolist()
    body_chunks = [bodies[i:i + CHUNK_SIZE] for i in range(0, total_len, CHUNK_SIZE)]
    
    all_body_ko = []
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(executor.map(translate_workflow, body_chunks), total=len(body_chunks), desc="번역 중"))
        for chunk in results:
            all_body_ko.extend(chunk)
            
    df['Review_ko'] = all_body_ko[:total_len]
    elapsed = time.time() - start_time

    # 저장
    print(f"\n💾 결과 저장 중: {OUTPUT_FILE}")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for record in df.to_dict(orient='records'):
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f"\n✨ 완료! 소요 시간: {elapsed/60:.1f}분")
    print(f"📊 처리 속도: {total_len/elapsed:.2f}건/초")

if __name__ == "__main__":
    main()

📥 Qoo10 데이터 로딩 중: ./qoo10_reviews_master.jsonl
✅ 총 2,834건 확인 (본문 번역만 진행)


번역 중: 100%|██████████| 567/567 [08:08<00:00,  1.16it/s]


💾 결과 저장 중: qoo10_master_translated_jp_ko.jsonl

✨ 완료! 소요 시간: 8.1분
📊 처리 속도: 5.81건/초
